# 06 — Minimal repro: vLLM `draft_model` cannot load compressed-tensors checkpoints

**Purpose**: a small, self-contained, publicly-runnable reproduction of the bug
found in `docs/findings.md` 2026-07-24 (`ValueError: ... weight_packed ...`) while
trying to use our own SGT-QAT Qwen3-1.7B checkpoint as a vLLM speculative-decoding
drafter for Qwen3-8B. That original repro used a private checkpoint (Google Drive)
and a mixed-precision (W4/W3) recipe.

**Update, 2026-07-26 — the plain-checkpoint simplification was wrong.** Section 3
(plain, single-scheme W4A16, no mixed precision) loaded successfully in vLLM — no
error. The bug is **not** "compressed-tensors checkpoints in general." Section 5
tests the actual remaining hypothesis: that it's specifically **mixed-precision**
(`config_groups` with different bit-widths per layer subset) compressed-tensors
checkpoints that fail. Do not assume section 5 will reproduce it either — run it
and see. `docs/vllm-bug-report-draft.md` must not be filed until one of these
cells actually reproduces the failure and the report's claim matches whichever
one it turns out to be.</cell id="cell-0">


## 1. Environment info for the bug report

Run this first — `docs/vllm-bug-report-draft.md` has placeholders for both outputs
below. Paste them back in exactly as printed, don't summarize/truncate.

In [ ]:
import os
!pip install -q vllm

# Known issue (docs/logs.md 2026-07-23, hit in notebooks 02/03): pip can resolve
# a CUDA-13-linked vLLM binary alongside a CUDA-12.x torch build -- `import vllm`
# then fails with `ImportError: libcudart.so.13`. Fix it here, before anything
# imports torch/vllm, in case this fresh environment hits the same mismatch.
import glob, subprocess
cu13_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/cu13/lib/libcudart.so.13')
if cu13_libs and not os.path.exists('/usr/lib/x86_64-linux-gnu/libcudart.so.13'):
    subprocess.run(['ln', '-sf', cu13_libs[0], '/usr/lib/x86_64-linux-gnu/libcudart.so.13'], check=True)
    subprocess.run(['ldconfig'], check=True)
    print(f"Symlinked {cu13_libs[0]} -> /usr/lib/x86_64-linux-gnu/libcudart.so.13 (CUDA runtime mismatch fix)")

In [ ]:
!pip show vllm
print("\n" + "="*80 + "\n")
!python -m vllm.collect_env

## 2. Build a tiny compressed-tensors checkpoint (plain W4A16 GPTQ, no mixed precision)

In [ ]:
!pip install -q llmcompressor compressed-tensors

import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MODEL_ID = 'Qwen/Qwen3-0.6B'  # small on purpose -- this repro doesn't need our 1.7B/8B pair
SEQ_LEN = 2048
CALIB_N = 32  # small on purpose -- this is a loading-path bug, not a quality benchmark
CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4a16-compressed-repro')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)

ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
samples, collected = [], 0
for item in ds:
    enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    if enc['input_ids'].shape[1] == SEQ_LEN:
        samples.append(enc['input_ids'])
        collected += 1
        if collected >= CALIB_N:
            break
calib_dataset = Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

# Plain W4A16, single scheme, no config_groups/mixed precision -- the simplest
# possible thing that still produces a genuinely packed compressed-tensors checkpoint.
recipe = GPTQModifier(targets='Linear', ignore=['lm_head'], scheme='W4A16', dampening_frac=0.01)
oneshot(model=model, dataset=calib_dataset, recipe=recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model.save_pretrained(str(CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

# Sanity check: confirm this genuinely saved packed weights, not a full-precision
# fallback -- look for weight_packed in the safetensors keys before even trying
# to load it into vLLM.
from safetensors import safe_open
shard = next(CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"Checkpoint saved to {CHECKPOINT_DIR}, contains weight_packed tensors: {has_packed}")
assert has_packed, "Checkpoint didn't actually save as compressed -- nothing to repro here."

del model
torch.cuda.empty_cache()

## 3. Trigger the bug: load it as a vLLM `draft_model`

Expected (per the original bug): this raises `ValueError: There is no module or
parameter named '...weight_packed' in ... . The available parameters belonging
to ... are: {'...weight'}`. **Actual result (2026-07-26): this did NOT raise —
the plain single-scheme checkpoint loaded successfully.** See the updated intro
cell and section 5 for the mixed-precision hypothesis this points to instead.</cell id="cell-6">


In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'  # so the real traceback surfaces here, not in a swallowed subprocess
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

llm = LLM(
    model=MODEL_ID,  # same tiny model as target, for simplicity -- the bug is in draft-model loading, not target/draft compatibility
    speculative_config={
        'method': 'draft_model',
        'model': str(CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

## 4. Copy the exact traceback from the cell above into `docs/vllm-bug-report-draft.md`

Paste the full traceback (not just the last line) into the "Actual behavior"
section, and the outputs from cell 1 into "Your current environment".

## 5. If section 3 did NOT raise an error: test the mixed-precision hypothesis

**Real result, 2026-07-26**: the plain single-scheme W4A16 checkpoint above
loaded successfully — no `ValueError`, full engine init completed. The minimal
repro did **not** reproduce the original bug.

This means the plain-W4A16 simplification likely removed the actual trigger.
Our real checkpoint used a **mixed-precision** recipe (`config_groups` with
*different* bit-widths for different layer subsets — W4 on protected layers,
W3 on the rest — see `notebooks/01`), not a single uniform scheme. That's a
structurally different compressed-tensors checkpoint (per-layer scheme
metadata, not one global scheme), and may be what vLLM's `draft_model` path
actually can't handle — not "compressed-tensors in general."

This cell tests that directly: same tiny model, same cheap calibration, but
with two `config_groups` (mirroring notebook 01's actual structure) instead of
one uniform scheme.

In [ ]:
import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MIXED_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro')
MIXED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_mixed = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)

# Reuse the same calibration approach as section 2 -- small on purpose.
ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
samples, collected = [], 0
for item in ds:
    enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    if enc['input_ids'].shape[1] == SEQ_LEN:
        samples.append(enc['input_ids'])
        collected += 1
        if collected >= CALIB_N:
            break
calib_dataset = Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

# Split layers arbitrarily (first half vs. second half by name order) -- unlike
# notebook 01, this isn't sensitivity-ranked, because that's irrelevant to the
# question here: does vLLM's draft_model loader choke on a checkpoint with TWO
# config_groups (different bit-widths per layer subset), regardless of which
# specific layers are in which group.
all_linear_names = [name for name, m in model_mixed.named_modules()
                     if isinstance(m, torch.nn.Linear) and name != 'lm_head']
mid = len(all_linear_names) // 2
group_w4, group_w3 = all_linear_names[:mid], all_linear_names[mid:]
print(f"{len(group_w4)} layers -> W4, {len(group_w3)} layers -> W3 (of {len(all_linear_names)} total)")

mixed_recipe = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'w4_group': {
            'targets': group_w4,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 4, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
        'w3_group': {
            'targets': group_w3,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 3, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_mixed, dataset=calib_dataset, recipe=mixed_recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model_mixed.save_pretrained(str(MIXED_CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(MIXED_CHECKPOINT_DIR))

from safetensors import safe_open
shard = next(MIXED_CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"Mixed-precision checkpoint saved to {MIXED_CHECKPOINT_DIR}, contains weight_packed tensors: {has_packed}")
assert has_packed, "Checkpoint didn't actually save as compressed -- nothing to repro here."

del model_mixed
torch.cuda.empty_cache()

### 5b. Trigger: load the mixed-precision checkpoint as a `draft_model`

**Restart the runtime before running this section** (Runtime -> Restart
session in Colab, then re-run the install cell + section 5's two cells, skip
section 3). vLLM's engine holds persistent CUDA context/state that may not
clean up cleanly between two separate `LLM()` instantiations in one process —
a fresh process avoids a false pass/fail caused by leftover state from
section 3's already-successful load, rather than the actual bug.

If this raises the `weight_packed` `ValueError` where section 3's plain
checkpoint didn't, that confirms the bug is specifically about mixed-precision
(`config_groups`) compressed-tensors checkpoints, not compressed-tensors
checkpoints in general — and `docs/vllm-bug-report-draft.md` needs rewriting
around that narrower, more precise claim before filing.

In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

llm_mixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

## 6. Verify the maintainer's fix (PR #49900)

Filed as [vllm-project/vllm#49893](https://github.com/vllm-project/vllm/issues/49893).
Maintainer `harjothkhara` root-caused it fast and matches our own finding:
the draft model loads under a `draft_model` prefix at runtime, which breaks
exact-name/anchored-regex `config_groups` target matching — single-scheme
worked because `Linear`-class-name matching is substring-based, mixed-precision
(name-based) targets weren't. Fix PR:
[#49900](https://github.com/vllm-project/vllm/pull/49900), Python-only (no
compile needed).

**Restart the runtime before this section** — installing a different vllm
build over the existing one, plus the same persistent-CUDA-context caution as
section 5b.

They asked us to confirm 4 things:
1. Our mixed-precision (`config_groups`) checkpoint now loads via
   `speculative_config={"method": "draft_model", ...}`.
2. The single-scheme W4A16 checkpoint still loads (no regression).
3. A short generation actually runs on the mixed-precision drafter (not just
   engine init).
4. The drafter's in-serving VRAM footprint vs. our decompressed workaround —
   confirms the compression actually survives loading, not just that loading
   no longer crashes.

In [ ]:
!pip uninstall -y vllm -q
!VLLM_USE_PRECOMPILED=1 pip install -q "git+https://github.com/harjothkhara/vllm.git@oss-find/vllm-2026-07-26"

!pip show vllm | grep -E "^(Name|Version)"

### 6a. Point 1 + setup for point 4: load the mixed-precision (compressed) checkpoint

Reuses `MIXED_CHECKPOINT_DIR` from section 5 (already on disk this session).
Records GPU memory before/after for the point-4 comparison later in this
section.

In [ ]:
import os, subprocess
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

def _gpu_memory_used_bytes(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024

baseline_mem_compressed = _gpu_memory_used_bytes()

from vllm import LLM

llm_mixed_fixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

loaded_mem_compressed = _gpu_memory_used_bytes()
delta_compressed = loaded_mem_compressed - baseline_mem_compressed
print(f"POINT 1: mixed-precision checkpoint loaded successfully: {llm_mixed_fixed is not None}")
print(f"In-serving memory delta (compressed mixed-precision drafter): {delta_compressed / 1024**3:.3f} GiB")

### 6b. Point 3: a short generation actually runs on the mixed-precision drafter

In [ ]:
from vllm import SamplingParams

outputs = llm_mixed_fixed.generate(
    ["The capital of France is"],
    SamplingParams(max_tokens=32, temperature=0),
)
for out in outputs:
    print("PROMPT:", out.prompt)
    print("OUTPUT:", out.outputs[0].text)

print(f"\nPOINT 3: generation completed without error: {len(outputs) > 0}")

### 6c. Point 4 (second half): compare against the decompressed workaround

Decompresses the same mixed-precision checkpoint the way `notebooks/03` had
to (before this fix existed), loads *that* as the drafter instead, and
compares its in-serving memory delta against section 6a's compressed-checkpoint
number. If the fix genuinely preserves compression through loading, the
compressed delta from 6a should be meaningfully smaller than this one.

**Restart the runtime before this cell** (same persistent-CUDA-context/two-`LLM()`
caution as before) — re-run the fix-install cell first, skip 6a/6b, land here.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from compressed_tensors import ModelCompressor
import torch.nn as nn

PLAIN_MIXED_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w3-mixed-plain-repro')

def _unwrap_quantized_linears(model):
    # same approach as notebooks/03 -- see docs/findings.md 2026-07-24
    replaced = 0
    for name, module in list(model.named_modules()):
        if hasattr(module, 'weight_scale') or hasattr(module, 'weight_shape'):
            parent_name, _, child_name = name.rpartition('.')
            parent = model.get_submodule(parent_name) if parent_name else model
            weight = module.weight.data
            has_bias = getattr(module, 'bias', None) is not None
            plain = nn.Linear(weight.shape[1], weight.shape[0], bias=has_bias, dtype=weight.dtype)
            plain.weight.data.copy_(weight)
            if has_bias:
                plain.bias.data.copy_(module.bias.data)
            setattr(parent, child_name, plain)
            replaced += 1
    print(f"Replaced {replaced} quantized modules with plain nn.Linear.")

if not PLAIN_MIXED_CHECKPOINT_DIR.exists():
    _decompress_model = AutoModelForCausalLM.from_pretrained(
        str(MIXED_CHECKPOINT_DIR), dtype=torch.float16, trust_remote_code=True
    )
    _compressor = ModelCompressor.from_pretrained_model(_decompress_model)
    _compressor.decompress_model(_decompress_model)
    _unwrap_quantized_linears(_decompress_model)
    if getattr(_decompress_model, 'hf_quantizer', None) is not None:
        _decompress_model.hf_quantizer.remove_quantization_config(_decompress_model)
    if hasattr(_decompress_model.config, 'quantization_config'):
        _decompress_model.config.quantization_config = None

    _decompress_tokenizer = AutoTokenizer.from_pretrained(str(MIXED_CHECKPOINT_DIR))
    PLAIN_MIXED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    _decompress_model.save_pretrained(str(PLAIN_MIXED_CHECKPOINT_DIR), save_original_format=False)
    _decompress_tokenizer.save_pretrained(str(PLAIN_MIXED_CHECKPOINT_DIR))
    del _decompress_model
    torch.cuda.empty_cache()

import os, subprocess
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

def _gpu_memory_used_bytes(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024

baseline_mem_plain = _gpu_memory_used_bytes()

from vllm import LLM

llm_plain_workaround = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(PLAIN_MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

loaded_mem_plain = _gpu_memory_used_bytes()
delta_plain = loaded_mem_plain - baseline_mem_plain
print(f"In-serving memory delta (decompressed/plain workaround drafter): {delta_plain / 1024**3:.3f} GiB")
print(f"\nPOINT 4: compare this against section 6a's compressed-checkpoint delta")
print(f"(re-run/reference that number here manually -- separate runtime session).")

### 6d. Point 2: confirm no regression on the single-scheme checkpoint

**Restart the runtime before this cell.** Re-run the fix-install cell first,
then this one — reuses `CHECKPOINT_DIR` from section 2 (already on disk).

In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

from vllm import LLM

llm_single_scheme_fixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

print(f"POINT 2: single-scheme checkpoint still loads (no regression): {llm_single_scheme_fixed is not None}")

## 7. Report back on the issue

Once sections 6a-6d are done, comment on
[vllm-project/vllm#49893](https://github.com/vllm-project/vllm/issues/49893)
with the four results (loaded: yes/no, generation output, and the two memory
deltas from 6a/6c). Real numbers only — same rule as everywhere else in this
project, don't summarize/round away the actual pasted output before posting.